# Lab 4: Spark SQL

## 🎯 **Learning Objectives:**
- Master Spark SQL for querying data
- Learn to use SQL syntax with Spark
- Understand CREATE TABLE, VIEW, and UDF
- Practice advanced SQL operations
- Learn catalog management

## 📚 **Key Concepts:**
1. **Spark SQL**: SQL interface for Spark
2. **CREATE TABLE**: Managed and external tables
3. **VIEW**: Virtual tables from queries
4. **UDF**: User Defined Functions
5. **Catalog**: Metadata management

## 🏗️ **Architecture Overview:**
```
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│   DataFrames    │───▶│   Spark SQL      │───▶│   SQL Results   │
│   (Python API)  │    │   (SQL Interface)│    │                 │
└─────────────────┘    └──────────────────┘    └─────────────────┘
         │                        │                        │
         ▼                        ▼                        ▼
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│ Register as     │    │ SQL Queries      │    │ Tables & Views  │
│ Temporary View  │    │ • SELECT         │    │ • Managed       │
│                 │    │ • JOIN          │    │ • External      │
│                 │    │ • Window Funcs  │    │ • Views         │
└─────────────────┘    └──────────────────┘    └─────────────────┘
```

## 📊 **Use Cases:**
- **SQL Queries**: Query DataFrames using SQL syntax
- **Table Management**: Create and manage tables
- **View Creation**: Create reusable views
- **Custom Functions**: Extend SQL with UDFs
- **Catalog Operations**: Manage database catalogs


In [1]:
import os
import sys

# Khởi tạo Spark Session cho Spark SQL (Cấu hình local bypass Docker errors)
try:
    from pyspark.sql import SparkSession
    from pyspark import SparkContext

    # Stop any existing SparkContext to avoid errors
    if SparkContext._active_spark_context is not None:
        SparkContext._active_spark_context.stop()

    spark = SparkSession.builder \
        .appName("Spark_SQL_Practice") \
        .master("local[*]") \
        .config("spark.driver.memory", "2g") \
        .config("spark.executor.memory", "2g") \
        .getOrCreate()
        
    print("🚀 Spark SQL Session initialized successfully!")
    print(f"📊 Spark Version: {spark.version}")
    print(f"🔗 Master URL: {spark.conf.get('spark.master')}")
except Exception as e:
    print(f"❌ Error initializing Spark: {e}")
    raise e

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/07 00:01:59 WARN Utils: Your hostname, son-ThinkBook-14-G2-ITL, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlp0s20f3)
26/04/07 00:01:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 00:02:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/07 00:02:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


🚀 Spark SQL Session initialized successfully!
📊 Spark Version: 4.1.1
🔗 Master URL: local[*]


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SparkSQLLab") \
    .master("spark://localhost:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

# Set log level to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

print("🚀 Spark SQL Session initialized successfully!")
print(f"📊 Spark Version: {spark.version}")
print(f"🔗 Master URL: {spark.sparkContext.master}")
print(f"💾 Default Catalog: {spark.catalog.currentCatalog()}")
print(f"📁 Current Database: {spark.catalog.currentDatabase()}")


🚀 Spark SQL Session initialized successfully!
📊 Spark Version: 4.1.1
🔗 Master URL: local[*]
💾 Default Catalog: spark_catalog
📁 Current Database: default


26/04/07 00:02:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Exercise 1: Basic Spark SQL Queries

### 🎯 **Learning Objectives:**
- Query DataFrames using SQL syntax
- Register DataFrames as temporary views
- Perform basic SQL operations
- Understand SQL vs DataFrame API


In [4]:
import random
from datetime import datetime, timedelta

print("📊 Creating sample datasets for Spark SQL Lab...")

# 1. Sales Dataset (Fact Table)
sales_data = []
customers = [f"User_{i}" for i in range(1, 101)]
products = ['Laptop', 'Smartphone', 'Tablet', 'Monitor', 'Headphones', 'Smartwatch']
categories = ['Electronics', 'Accessories', 'Computing']

for i in range(1000):
    sales_data.append({
        'sale_id': f'SALE_{i+1:04d}',
        'customer_name': random.choice(customers),
        'product_name': random.choice(products),
        'category': random.choice(categories),
        'quantity': random.randint(1, 5),
        'unit_price': round(random.uniform(50, 2000), 2),
        'sale_date': (datetime.now() - timedelta(days=random.randint(0, 365))).strftime('%Y-%m-%d'),
        'region': random.choice(['North', 'South', 'East', 'West', 'Central'])
    })

# Sample Customer Data
customer_data = []
for c in set(random.choice(customers) for _ in range(50)): # Subset of customers
    customer_data.append({
        'customer_name': c,
        'email': f"{c.lower()}@example.com",
        'signup_date': (datetime.now() - timedelta(days=random.randint(30, 1000))).strftime('%Y-%m-%d'),
        'loyalty_tier': random.choice(['Bronze', 'Silver', 'Gold', 'Platinum', 'Diamond'])
    })

print("\n🎯 Creating DataFrames and Temp Views...")
sales_df = spark.createDataFrame(sales_data)
customers_df = spark.createDataFrame(customer_data)

# Show data structure
print("\nSales DataFrame Schema:")
sales_df.printSchema()

print("Customers DataFrame Schema:")
customers_df.printSchema()

📊 Creating sample datasets for Spark SQL Lab...

🎯 Creating DataFrames and Temp Views...

Sales DataFrame Schema:
root
 |-- category: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- unit_price: double (nullable = true)

Customers DataFrame Schema:
root
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: string (nullable = true)



In [5]:
# Register DataFrames as Temporary Views
print("📝 Registering DataFrames as temporary views for SQL queries...")

sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")

print("✅ Views created:")
print("   - sales")
print("   - customers")

# List all temporary views
print("\n📋 Available temporary views:")
spark.sql("SHOW TABLES").show()


📝 Registering DataFrames as temporary views for SQL queries...
✅ Views created:
   - sales
   - customers

📋 Available temporary views:
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |customers|       true|
|         |    sales|       true|
+---------+---------+-----------+



In [6]:
# Basic SQL Queries
print("🔍 Exercise 1: Basic Spark SQL Queries")
print("=" * 60)

print("\n1️⃣ Simple SELECT query:")
result1 = spark.sql("""
    SELECT 
        sale_id,
        customer_name,
        product_name,
        quantity,
        unit_price,
        (quantity * unit_price) as total_amount
    FROM sales
    LIMIT 10
""")
result1.show(truncate=False)

print("\n2️⃣ Filtering with WHERE clause:")
result2 = spark.sql("""
    SELECT *
    FROM sales
    WHERE category = 'Electronics'
        AND quantity > 2
    ORDER BY unit_price DESC
    LIMIT 10
""")
result2.show(truncate=False)

print("\n3️⃣ Aggregations with GROUP BY:")
result3 = spark.sql("""
    SELECT 
        product_name,
        COUNT(*) as sale_count,
        SUM(quantity) as total_quantity,
        AVG(unit_price) as avg_price,
        SUM(quantity * unit_price) as total_revenue
    FROM sales
    GROUP BY product_name
    ORDER BY total_revenue DESC
""")
result3.show(truncate=False)


🔍 Exercise 1: Basic Spark SQL Queries

1️⃣ Simple SELECT query:


+---------+-------------+------------+--------+----------+-----------------+
|sale_id  |customer_name|product_name|quantity|unit_price|total_amount     |
+---------+-------------+------------+--------+----------+-----------------+
|SALE_0001|User_66      |Smartwatch  |2       |1928.59   |3857.18          |
|SALE_0002|User_98      |Monitor     |1       |683.94    |683.94           |
|SALE_0003|User_1       |Smartphone  |1       |1919.55   |1919.55          |
|SALE_0004|User_31      |Tablet      |2       |1928.92   |3857.84          |
|SALE_0005|User_72      |Monitor     |2       |749.8     |1499.6           |
|SALE_0006|User_46      |Smartphone  |2       |1122.87   |2245.74          |
|SALE_0007|User_74      |Smartwatch  |5       |1597.64   |7988.200000000001|
|SALE_0008|User_41      |Smartwatch  |4       |62.81     |251.24           |
|SALE_0009|User_91      |Smartwatch  |1       |1946.35   |1946.35          |
|SALE_0010|User_31      |Smartwatch  |2       |1075.37   |2150.74          |

## Exercise 2: JOIN Operations with SQL

### 🎯 **Learning Objectives:**
- Perform JOINs using SQL syntax
- Understand different JOIN types
- Combine multiple tables


In [8]:
print("\n🔗 Exercise 2: JOIN Operations with SQL")
print("=" * 60)

print("\n1️⃣ INNER JOIN:")
result4 = spark.sql("""
    SELECT 
        s.sale_id,
        s.customer_name,
        s.product_name,
        s.quantity,
        s.unit_price,
        c.email,
        c.loyalty_tier,
        c.signup_date
    FROM sales s
    INNER JOIN customers c
        ON s.customer_name = c.customer_name
    LIMIT 10
""")
result4.show(truncate=False)

print("\n2️⃣ LEFT JOIN:")
print("Showing sales that might not have matching customer details (Guest purchases)")
result5 = spark.sql("""
    SELECT 
        s.sale_id,
        s.customer_name,
        s.product_name,
        c.loyalty_tier
    FROM sales s
    LEFT JOIN customers c
        ON s.customer_name = c.customer_name
    LIMIT 10
""")
result5.show(truncate=False)

print("\n3️⃣ Analytical query using JOIN: Total revenue by loyalty tier")
result6 = spark.sql("""
    SELECT 
        COALESCE(c.loyalty_tier, 'Guest/No Account') as membership_level,
        COUNT(s.sale_id) as total_purchases,
        ROUND(SUM(s.quantity * s.unit_price), 2) as total_revenue,
        ROUND(AVG(s.quantity * s.unit_price), 2) as avg_order_value
    FROM sales s
    LEFT JOIN customers c
        ON s.customer_name = c.customer_name
    GROUP BY c.loyalty_tier
    ORDER BY total_revenue DESC
""")
result6.show(truncate=False)


🔗 Exercise 2: JOIN Operations with SQL

1️⃣ INNER JOIN:
+---------+-------------+------------+--------+----------+-------------------+------------+-----------+
|sale_id  |customer_name|product_name|quantity|unit_price|email              |loyalty_tier|signup_date|
+---------+-------------+------------+--------+----------+-------------------+------------+-----------+
|SALE_0997|User_81      |Laptop      |5       |457.77    |user_81@example.com|Gold        |2024-09-03 |
|SALE_0966|User_81      |Headphones  |5       |1072.93   |user_81@example.com|Gold        |2024-09-03 |
|SALE_0960|User_81      |Smartwatch  |3       |1358.8    |user_81@example.com|Gold        |2024-09-03 |
|SALE_0957|User_81      |Tablet      |1       |1157.24   |user_81@example.com|Gold        |2024-09-03 |
|SALE_0852|User_81      |Smartphone  |2       |337.56    |user_81@example.com|Gold        |2024-09-03 |
|SALE_0773|User_81      |Laptop      |2       |508.28    |user_81@example.com|Gold        |2024-09-03 |
|SALE_0

In [9]:
# Window Functions with SQL
print("🪟 Exercise 3: Window Functions with SQL")
print("=" * 60)

print("\n1️⃣ ROW_NUMBER and RANK:")
result7 = spark.sql("""
    SELECT 
        customer_name,
        product_name,
        quantity * unit_price as total_amount,
        ROW_NUMBER() OVER (
            PARTITION BY customer_name 
            ORDER BY quantity * unit_price DESC
        ) as row_num,
        RANK() OVER (
            PARTITION BY customer_name 
            ORDER BY quantity * unit_price DESC
        ) as rank,
        DENSE_RANK() OVER (
            PARTITION BY customer_name 
            ORDER BY quantity * unit_price DESC
        ) as dense_rank
    FROM sales
    WHERE customer_name = 'Alice'
    ORDER BY total_amount DESC
""")
result7.show(truncate=False)

print("\n2️⃣ Running Totals:")
result8 = spark.sql("""
    SELECT 
        sale_id,
        customer_name,
        sale_date,
        quantity * unit_price as transaction_amount,
        SUM(quantity * unit_price) OVER (
            PARTITION BY customer_name 
            ORDER BY sale_date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) as running_total
    FROM sales
    WHERE customer_name = 'Alice'
    ORDER BY sale_date
    LIMIT 10
""")
result8.show(truncate=False)

print("\n3️⃣ Moving Average:")
result9 = spark.sql("""
    SELECT 
        sale_date,
        product_name,
        quantity * unit_price as daily_revenue,
        AVG(quantity * unit_price) OVER (
            PARTITION BY product_name 
            ORDER BY sale_date 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) as moving_avg_3days
    FROM sales
    WHERE product_name = 'Laptop'
    ORDER BY sale_date
    LIMIT 10
""")
result9.show(truncate=False)


🪟 Exercise 3: Window Functions with SQL

1️⃣ ROW_NUMBER and RANK:
+-------------+------------+------------+-------+----+----------+
|customer_name|product_name|total_amount|row_num|rank|dense_rank|
+-------------+------------+------------+-------+----+----------+
+-------------+------------+------------+-------+----+----------+


2️⃣ Running Totals:
+-------+-------------+---------+------------------+-------------+
|sale_id|customer_name|sale_date|transaction_amount|running_total|
+-------+-------------+---------+------------------+-------------+
+-------+-------------+---------+------------------+-------------+


3️⃣ Moving Average:
+----------+------------+-------------+------------------+
|sale_date |product_name|daily_revenue|moving_avg_3days  |
+----------+------------+-------------+------------------+
|2025-04-07|Laptop      |970.36       |970.36            |
|2025-04-08|Laptop      |189.28       |579.82            |
|2025-04-18|Laptop      |3649.96      |1603.2            |
|202

In [11]:
print("\n👁️ Exercise 4: CREATE VIEW")
print("=" * 60)

print("\n1️⃣ Create a temporary view for customer sales summary:")
spark.sql("""
    CREATE OR REPLACE TEMP VIEW customer_sales_summary AS
    SELECT 
        c.customer_name,
        c.email,
        c.loyalty_tier,
        COUNT(s.sale_id) as total_orders,
        SUM(s.quantity * s.unit_price) as total_spent,
        AVG(s.quantity * s.unit_price) as avg_order_value,
        MAX(s.sale_date) as last_purchase_date
    FROM customers c
    LEFT JOIN sales s
        ON c.customer_name = s.customer_name
    GROUP BY c.customer_name, c.email, c.loyalty_tier
""")

print("✅ View 'customer_sales_summary' created!")

# Query the view
print("\nTop 5 customers by total spent from view:")
spark.sql("""
    SELECT * 
    FROM customer_sales_summary 
    ORDER BY total_spent DESC NULLS LAST
    LIMIT 5
""").show(truncate=False)

print("\n2️⃣ Create another view for product performance:")
spark.sql("""
    CREATE OR REPLACE TEMP VIEW product_performance AS
    SELECT 
        product_name,
        category,
        COUNT(sale_id) as times_sold,
        SUM(quantity) as total_quantity_sold,
        MIN(unit_price) as min_price_sold,
        MAX(unit_price) as max_price_sold,
        ROUND(SUM(quantity * unit_price), 2) as total_revenue
    FROM sales
    GROUP BY product_name, category
""")

print("✅ View 'product_performance' created!")

print("\nProduct performance overview:")
spark.sql("""
    SELECT * 
    FROM product_performance 
    ORDER BY total_revenue DESC
""").show(truncate=False)

print("\n📋 List all active temporary views:")
spark.sql("SHOW VIEWS").show(truncate=False)


👁️ Exercise 4: CREATE VIEW

1️⃣ Create a temporary view for customer sales summary:
✅ View 'customer_sales_summary' created!

Top 5 customers by total spent from view:


+-------------+-------------------+------------+------------+------------------+------------------+------------------+
|customer_name|email              |loyalty_tier|total_orders|total_spent       |avg_order_value   |last_purchase_date|
+-------------+-------------------+------------+------------+------------------+------------------+------------------+
|User_74      |user_74@example.com|Gold        |16          |73989.07          |4624.316875       |2026-02-07        |
|User_53      |user_53@example.com|Gold        |16          |69984.56999999999 |4374.0356249999995|2026-03-25        |
|User_23      |user_23@example.com|Gold        |12          |58796.219999999994|4899.6849999999995|2026-03-20        |
|User_46      |user_46@example.com|Gold        |14          |42919.50999999999 |3065.679285714285 |2026-03-04        |
|User_41      |user_41@example.com|Diamond     |13          |38589.380000000005|2968.4138461538464|2026-02-20        |
+-------------+-------------------+------------+

In [13]:
print("\n📊 Exercise 5: CREATE TABLE")
print("=" * 60)

print("\n1️⃣ Create a managed table from query result:")
# Drop table if exists to simulate REPLACE behavior
spark.sql("DROP TABLE IF EXISTS sales_summary")
spark.sql("""
    CREATE TABLE sales_summary AS
    SELECT 
        region,
        category,
        COUNT(*) as total_sales,
        SUM(quantity) as total_quantity,
        SUM(quantity * unit_price) as total_revenue,
        AVG(quantity * unit_price) as avg_transaction_value
    FROM sales
    GROUP BY region, category
""")

print("✅ Managed table 'sales_summary' created!")

# Query the table
print("\nQuerying 'sales_summary' table:")
spark.sql("SELECT * FROM sales_summary ORDER BY total_revenue DESC LIMIT 5").show(truncate=False)

print("\n2️⃣ Describe table details:")
spark.sql("DESCRIBE FORMATTED sales_summary").show(truncate=False)

print("\n3️⃣ Show tables in current database:")
spark.sql("SHOW TABLES").show(truncate=False)

print("\n4️⃣ Extract data using CTE and insert to new table:")
spark.sql("DROP TABLE IF EXISTS high_value_customers")
spark.sql("""
    CREATE TABLE high_value_customers AS
    WITH customer_stats AS (
        SELECT 
            customer_name,
            COUNT(sale_id) as total_purchases,
            SUM(quantity * unit_price) as total_spend
        FROM sales
        GROUP BY customer_name
    )
    SELECT * FROM customer_stats WHERE total_spend > 5000
""")
print("✅ Table 'high_value_customers' created!")
spark.sql("SELECT * FROM high_value_customers ORDER BY total_spend DESC LIMIT 5").show()


📊 Exercise 5: CREATE TABLE

1️⃣ Create a managed table from query result:
✅ Managed table 'sales_summary' created!

Querying 'sales_summary' table:
+-------+-----------+-----------+--------------+------------------+---------------------+
|region |category   |total_sales|total_quantity|total_revenue     |avg_transaction_value|
+-------+-----------+-----------+--------------+------------------+---------------------+
|East   |Computing  |76         |243           |254100.97000000006|3343.4338157894745   |
|Central|Computing  |78         |239           |251729.83000000005|3227.3055128205133   |
|East   |Electronics|63         |218           |244338.78000000003|3878.393333333334    |
|West   |Accessories|72         |217           |230573.43000000002|3202.4087500000005   |
|South  |Computing  |69         |209           |225224.09         |3264.1172463768116   |
+-------+-----------+-----------+--------------+------------------+---------------------+


2️⃣ Describe table details:
+----------

## Exercise 6: User Defined Functions (UDF)

### 🎯 **Learning Objectives:**
- Create User Defined Functions (UDFs)
- Register UDFs for use in SQL
- Understand UDF performance considerations


In [15]:
print("\n⚙️ Exercise 6: User Defined Functions (UDF)")
print("=" * 60)

print("\n1️⃣ Create a simple UDF:")
# Define Python function
def apply_discount(amount):
    if amount > 1000:
        return amount * 0.10
    elif amount > 500:
        return amount * 0.05
    return 0.0

# Register as SQL UDF
spark.udf.register("calculate_discount", apply_discount)
print("✅ UDF 'calculate_discount' registered!")

# Use UDF in SQL
result14 = spark.sql("""
    SELECT 
        sale_id,
        customer_name,
        quantity * unit_price as total_amount,
        calculate_discount(quantity * unit_price) as discount,
        (quantity * unit_price) - calculate_discount(quantity * unit_price) as final_amount
    FROM sales
    WHERE quantity * unit_price > 500
    ORDER BY total_amount DESC
    LIMIT 10
""")
result14.show(truncate=False)

print("\n2️⃣ Create a UDF for string manipulation:")
def format_customer_name(name, level):
    if not level:
        level = "Guest"
    return f"🌟 [{level.upper()}] - {name}"

spark.udf.register("format_customer", format_customer_name)
print("✅ UDF 'format_customer' registered!")

# Use UDF in SQL
result15 = spark.sql("""
    SELECT 
        format_customer(c.customer_name, c.loyalty_tier) as formatted_name,
        COUNT(s.sale_id) as total_orders,
        SUM(s.quantity * s.unit_price) as total_spent
    FROM customers c
    LEFT JOIN sales s
        ON c.customer_name = s.customer_name
    GROUP BY c.customer_name, c.loyalty_tier
    ORDER BY total_spent DESC
""")
result15.show(truncate=False)


⚙️ Exercise 6: User Defined Functions (UDF)

1️⃣ Create a simple UDF:
✅ UDF 'calculate_discount' registered!


26/04/07 00:04:21 WARN SimpleFunctionRegistry: The function calculate_discount replaced a previously registered function.
26/04/07 00:04:22 WARN SimpleFunctionRegistry: The function format_customer replaced a previously registered function.


+---------+-------------+-----------------+-----------------+-----------------+
|sale_id  |customer_name|total_amount     |discount         |final_amount     |
+---------+-------------+-----------------+-----------------+-----------------+
|SALE_0359|User_15      |9840.15          |984.015          |8856.135         |
|SALE_0611|User_13      |9669.6           |966.96           |8702.64          |
|SALE_0393|User_4       |9587.35          |958.7350000000001|8628.615         |
|SALE_0047|User_74      |9515.05          |951.505          |8563.545         |
|SALE_0013|User_75      |9480.800000000001|948.0800000000002|8532.720000000001|
|SALE_0446|User_18      |9427.9           |942.79           |8485.11          |
|SALE_0375|User_36      |9426.3           |942.63           |8483.67          |
|SALE_0240|User_37      |9365.650000000001|936.5650000000002|8429.085000000001|
|SALE_0877|User_54      |9349.550000000001|934.9550000000002|8414.595000000001|
|SALE_0867|User_23      |9303.5         

## Exercise 7: Catalog Management

### 🎯 **Learning Objectives:**
- Understand Spark catalog system
- List databases, tables, and functions
- Manage database and table metadata


In [16]:
# Catalog Management
print("🗂️ Exercise 7: Catalog Management")
print("=" * 60)

print("\n1️⃣ List all databases:")
spark.sql("SHOW DATABASES").show()

print("\n2️⃣ Show current database:")
print(f"Current database: {spark.catalog.currentDatabase()}")

print("\n3️⃣ List all tables in current database:")
spark.sql("SHOW TABLES").show()

print("\n4️⃣ Describe a table:")
spark.sql("DESCRIBE EXTENDED sales_summary").show(truncate=False)

print("\n5️⃣ List all functions:")
print("Available functions (first 20):")
spark.sql("SHOW FUNCTIONS").show(20, truncate=False)

print("\n6️⃣ List user-defined functions:")
print("User-defined functions:")
spark.sql("SHOW USER FUNCTIONS").show()

print("\n7️⃣ Get table details using catalog API:")
tables = spark.catalog.listTables()
print(f"\nTotal tables: {len(tables)}")
for table in tables:
    print(f"  - {table.name} (type: {table.tableType})")


🗂️ Exercise 7: Catalog Management

1️⃣ List all databases:
+---------+
|namespace|
+---------+
|  default|
+---------+


2️⃣ Show current database:
Current database: default

3️⃣ List all tables in current database:
+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|high_value_customers|      false|
|  default|       sales_summary|      false|
|         |customer_sales_su...|       true|
|         |           customers|       true|
|         | product_performance|       true|
|         |               sales|       true|
+---------+--------------------+-----------+


4️⃣ Describe a table:
+----------------------------+------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                         

## Summary

### ✅ What we learned:
1. **Spark SQL Basics**: Query DataFrames using SQL syntax
2. **JOIN Operations**: INNER JOIN, LEFT JOIN with SQL
3. **Window Functions**: ROW_NUMBER, RANK, running totals, moving averages
4. **CREATE VIEW**: Create reusable temporary views
5. **CREATE TABLE**: Create managed tables from queries
6. **User Defined Functions**: Create and register UDFs for SQL
7. **Catalog Management**: Manage databases, tables, and functions

### 🎯 Key Takeaways:
- Spark SQL provides a familiar SQL interface for Spark
- Views simplify complex queries and improve reusability
- UDFs extend SQL functionality with custom logic
- Catalog API helps manage metadata and discover resources

### 🚀 Next Steps:
- Practice with more complex SQL queries
- Explore advanced SQL features (CTEs, subqueries)
- Learn about Spark SQL performance optimization
- Integrate Spark SQL with external data sources
